In [ ]:
import PyPhenom as ppi
import numpy as np
import time
import os
import pandas as pd

# Phenom connection credentials
PhenomID = ######
username = ######
password =######

# Connect to the Phenom
phenom = ppi.Phenom(PhenomID, username, password)

# Folder setup for data
base_path = r'X:\\'
nested_folder_name = ######
nested_folder_path = os.path.join(base_path, nested_folder_name)
os.makedirs(nested_folder_path, exist_ok=True)
os.chdir(nested_folder_path)

# Initialize an empty dictionary to store coordinates
stage_coordinates = {}


# Capture Top Left corner

In [ ]:
# Get current stage coordinates
position = phenom.GetStageModeAndPosition().position
stage_coordinates['Top Left'] = {'x': position.x, 'y': position.y, 'z': phenom.GetSemWD()}
print("Top Left Corner:", stage_coordinates['Top Left'])


# Capture Top Right Corner

In [ ]:
position = phenom.GetStageModeAndPosition().position
stage_coordinates['Top Right'] = {'x': position.x, 'y': position.y, 'z': phenom.GetSemWD()}
print("Top Right Corner:", stage_coordinates['Top Right'])


# Coordinates to csv


In [ ]:
# Convert dictionary to DataFrame and save as CSV
df = pd.DataFrame(stage_coordinates).T  # .T flips the dictionary for rows as labels
csv_path = os.path.join(nested_folder_path, 'SEM_stage_coordinates_Nov24.csv')
df.to_csv(csv_path, index=True)  # Keep index so corner names are preserved
print(f"Coordinates exported to: {csv_path}")


# Run the cell below to locate the start of the image acquisition.

In [ ]:
x_loc_1 = phenom.GetStageModeAndPosition().position.x
y_loc_1 = phenom.GetStageModeAndPosition().position.y
z_loc_1 = phenom.GetSemWD()

# Running the cell below initiates the image acquisition loop

In [ ]:
#phenom.SetSemHighTension(-10000) #Acceleration Voltage = 10kV

acqScanParams = ppi.ScanParams()
acqScanParams.size = ppi.Size(1920, 1200)
acqScanParams.detector = ppi.DetectorMode.Mix
acqScanParams.nFrames = 16
acqScanParams.hdr= False
acqScanParams.scale = 1.0
#acq = phenom.SemAcquireImage(acqScanParams)
#ppi.Save(acq, 'example5um.tiff')

sample = nested_folder_name


## How far do you want to scan, in meters
x_length = .001
y_length = .0005

field_width = 50e-6
phenom.SetHFW(field_width)

x_image_numbers = round( (x_length / field_width))
y_image_numbers = round( (y_length / field_width *1920/1200))

phenom.MoveTo(x_loc_1, y_loc_1)
time.sleep(10)

#def acquire_image(acqScanParam,sample,equation_array):
def acquire_image(acqScanParam,sample):
        acq = phenom.SemAcquireImage(acqScanParams)
        ppi.Save(acq, sample + '_y_'+ str(y_location) + '_x_' + str(x_location) + ".tiff")

for i in range(y_image_numbers):
    for j in range (x_image_numbers):
        x_location = phenom.GetStageModeAndPosition().position.x 
        y_location = phenom.GetStageModeAndPosition().position.y
        phenom.SemAutoContrastBrightness()
        time.sleep(1)
        phenom.SemAutoFocus(ppi.AutoFocusAlgorithm.FineOnly) 
        z_focus = phenom.GetSemWD()
        count = 0
        if z_focus > .0082 or z_focus < 0.0065:
            print('trying again')
            phenom.SetSemWD(.007)
            phenom.SemAutoFocus(ppi.AutoFocusAlgorithm.FineOnly) 
            count = count + 1
            if count > 3:
                break
        #acquire_image(acqScanParams,sample,equation_array)
        acquire_image(acqScanParams,sample)
        phenom.MoveBy(field_width,0)
        time.sleep(6)
    acquire_image(acqScanParams,sample)
    phenom.MoveTo(x_loc_1, y_loc_1 + (i + 1) * field_width * 1920 / 1200)
